# MQTBench suite for Graphix simulators

To run this notebook, install the package with extra dependencies:

```bash
uv sync --extra examples
```

#### Benchmark characterization

As of version 0.3.5, Graphix supports two optimizations at the pattern level: _space minimization_ and _Pauli removal_.

- Space minimization rearranges the pattern commands in order to minimize the maximum number of qubits alive at any given time during the execution. This optimization is crucial to reduce the memory allocation in dense-state simulations. For patterns with causal flow (e.g., those directly transpiled from a quantum circuit), `Pattern.minimize_space` returns an optimal vale: `max_space = n_qubits + 1`.

- Pauli removal removes Pauli measurements on non-input qubits at the expense of adding local Clifford commands. This optimization can significantly reduce the number of commands (and, specially, measurement commands which are the bottleneck in dense-state simulations). However, it comes with a trade-off: patterns with causal flow are only guaranteed to have gflow after this optimization step. Performing space minimization on patterns without causal flow is known to be an NP-hard problem, and heuristics can fail to find a "good" measurement order.

As show in the table below, "Pauli removal + Space minimization" will often reduce the number of commands but `max_space` can become significantly larger than if only "Space minimization" is applied.

In [3]:
from graphix_mqtbench import generate_benchmarks
import pandas as pd

nqubit = 4
benchmarks = generate_benchmarks(nqubit)

rows = []
for bench in benchmarks:
    p = bench.pattern
    p_space = p.minimize_space(copy=True)
    p_pauli = p.infer_pauli_measurements().remove_pauli_measurements(copy=True)
    p_pauli_space = p_pauli.minimize_space(copy=True)

    rows.append({
        ("Circuit", "Benchmark"): bench.name.value,
        ("Circuit", "Qubits"): bench.nqubits,
        ("Circuit", "Gates"): len(bench.circuit.instruction),
        ("Transpilation", "Max Space"): p.max_space(),
        ("Transpilation", "Cmds"): len(p),
        ("Space min.", "Max Space"): p_space.max_space(),
        ("Space min.", "Cmds"): len(p_space),
        ("Pauli removal", "Max Space"): p_pauli.max_space(),
        ("Pauli removal", "Cmds"): len(p_pauli),
        ("Pauli removal + Space min.", "Max Space"): p_pauli_space.max_space(),
        ("Pauli removal + Space min.", "Cmds"): len(p_pauli_space),
    })

df = pd.DataFrame(rows)
df.columns = pd.MultiIndex.from_tuples(df.columns)
df

Circuit              Transpilation       Space min.        \
                  Benchmark Qubits Gates     Max Space  Cmds  Max Space  Cmds   
0                        ae      4   119             5   716          5   716   
1     bmw_quark_cardinality      4   123             5   656          5   656   
2          bmw_quark_copula      4    72             5   396          5   396   
3                        bv      4     4             5    17          5    17   
4   cdkm_ripple_carry_adder      4     7             5   235          5   235   
5                        dj      4    16             5    89          5    89   
6          draper_qft_adder      4    29             5   180          5   180   
7                full_adder      4     6             5   228          5   228   
8                       ghz      4     4             5    32          5    32   
9                graphstate      4     8             5    24          5    24   
10                   grover      4   132             5   939          5   939   
11                      hhl      4    49             5   306          5   306   
12            modular_adder      4    31             5   180          5   180   
13               multiplier      4    43             5   397          5   397   
14                     qaoa      4    24             5   151          5   151   
15                      qft      4    34             5   212          5   212   
16             qftentangled      4    38             5   236          5   236   
17                      qnn      4    27             5   197          5   197   
18                 qpeexact      4    25             5   154          5   154   
19               qpeinexact      4    37             5   224          5   224   
20                    qwalk      4   250             5  2135          5  2135   
21            randomcircuit      4   108             5   968          5   968   
22        rg_qft_multiplier      4    40             5   252          5   252   
23   vbe_ripple_carry_adder      4     6             5   228          5   228   
24             vqe_real_amp      4    25             5   263          5   263   
25                  vqe_su2      4    89             5   551          5   551   
26            vqe_two_local      4    34             5   326          5   326   
27                   wstate      4    13             5   110          5   110   

   Pauli removal      Pauli removal + Space min.       
       Max Space Cmds                  Max Space Cmds  
0             23  183                         10  183  
1             12  167                         10  167  
2             12  145                         12  145  
3              6   11                          5   11  
4             20   99                         20   99  
5              5   23                          5   23  
6             13   54                          9   54  
7             14   74                         12   74  
8              5   28                          5   28  
9              5   24                          5   24  
10            19  309                         19  309  
11            16   94                         16   94  
12            12   57                          9   57  
13            22  123                          8  123  
14             8   66                          8   66  
15            17   79                         11   79  
16            13   98                          9   98  
17             6   66                          5   66  
18             9   51                          9   51  
19            17   90                          8   90  
20           117  695                        109  695  
21            45  299                         24  299  
22            15   96                         13   96  
23            14   74                         12   74  
24             8  112                          8  112  
25            11  167                          8  167  
26            10  110

#### Minimal backend benchmark

In [2]:
import timeit
from graphix_mqtbench import MQTBenchmark, BenchmarkName
import numpy as np

rng = np.random.default_rng(42)

def simulate(pattern, backend):
    def run():
        return pattern.simulate_pattern(backend=backend, rng=rng)
    return run

benchmark = MQTBenchmark(name=BenchmarkName.QFT, nqubits=14)
pattern = benchmark.pattern.minimize_space()

run = simulate(pattern, backend="statevector")
timer = timeit.Timer(run)
t = min(timer.repeat(number=1, repeat=5))

print(
f"Benchmark = {benchmark.name.value}\n\
nqubits = {benchmark.nqubits}\n\
max_space = {pattern.max_space()}\n\
n_commands = {len(pattern)}\n\
simulation time = {t:.5f} s")

Benchmark = qft
nqubits = 14
max_space = 15
n_commands = 2982
simulation time = 0.65946 s
